# 🌡️ Thermal & Electrical Anomalies — Question 2b
**How can time-series visualizations highlight subtle deviations in multi-sensor data?**

### Approach
Raw sensor data is too noisy to reveal subtle pre-fault patterns.
We apply three progressive visualization techniques:
1. Multi-sensor overlay with rolling means
2. Z-score standardization — puts all sensors on the same scale
3. Anomaly heatmap — shows which sensors deviate simultaneously

### Sections
1. Setup & Data Loading
2. Farm A — Multi-Sensor Deviation Analysis
3. Farm B — Multi-Sensor Deviation Analysis
4. Farm C — Multi-Sensor Deviation Analysis
5. Cross-Farm Anomaly Heatmap Comparison

---
## Section 1 — Setup & Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

plt.rcParams['figure.figsize'] = (15, 5)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

print('✅ Imports OK')

In [ ]:
# ⚠️ UPDATE BASE PATH
BASE = Path('/Users/prabhaseessingh/Desktop/Data Science/DSMLCFinalComp/CARE_To_Compare')

FARM_A = BASE / 'Wind Farm A'
FARM_B = BASE / 'Wind Farm B'
FARM_C = BASE / 'Wind Farm C'

def load_farm(farm_path):
    return {
        'event_info': pd.read_csv(farm_path / 'event_info.csv', sep=';'),
        'feat_desc':  pd.read_csv(farm_path / 'feature_description.csv', sep=';'),
        'datasets':   farm_path / 'datasets'
    }

def get_label(feat_desc, bare_name):
    d = feat_desc[feat_desc['sensor_name'] == bare_name]['description'].values
    return d[0] if len(d) > 0 else bare_name

def load_event(farm, event_id):
    df = pd.read_csv(farm['datasets'] / f'{event_id}.csv', sep=';')
    return df.sort_values('id').reset_index(drop=True)

def rolling_zscore(series, window=1008):
    roll = series.rolling(window=window, min_periods=window//2)
    return (series - roll.mean()) / (roll.std() + 1e-9)

farms = {
    'A': load_farm(FARM_A),
    'B': load_farm(FARM_B),
    'C': load_farm(FARM_C)
}

print('✅ Farms loaded')

---
## Section 2 — Farm A: Multi-Sensor Deviation Analysis
Transformer failure event (event 68) — transformer phase temperatures + generator sensors.

In [ ]:
farm_a    = farms['A']
feat_a    = farm_a['feat_desc']
df_a      = load_event(farm_a, 68)
train_end_a = df_a[df_a['train_test'] == 'train'].index.max()

SENSORS_A = {
    'sensor_38': 'sensor_38_avg',
    'sensor_39': 'sensor_39_avg',
    'sensor_40': 'sensor_40_avg',
    'sensor_15': 'sensor_15_avg',
    'sensor_16': 'sensor_16_avg',
}

print('Farm A sensors for 2b analysis:')
for bare, col in SENSORS_A.items():
    print(f'  {col}: {get_label(feat_a, bare)}')

In [ ]:
# Plot 1: Rolling mean overlay — all sensors on same chart
DRIFT_WINDOW = 1008
colors_a = ['tomato', 'darkorange', 'purple', 'steelblue', 'seagreen']

fig, ax = plt.subplots(figsize=(15, 6))

for (bare, col), color in zip(SENSORS_A.items(), colors_a):
    if col not in df_a.columns:
        continue
    label   = get_label(feat_a, bare)
    rolling = df_a[col].rolling(window=DRIFT_WINDOW, min_periods=DRIFT_WINDOW//2).mean()
    ax.plot(df_a.index, rolling, linewidth=1.8, color=color,
            label=f'{label[:30]} ({col})')

ax.axvline(x=train_end_a, color='black', linestyle='--',
           linewidth=1.5, label='Fault window start')
ax.set_title('Farm A — Transformer Failure: 7-Day Rolling Mean\nMultiple Thermal Sensors Overlaid',
             fontweight='bold')
ax.set_xlabel('Time step (10-min intervals)')
ax.set_ylabel('Temperature (°C)')
ax.legend(fontsize=8, loc='upper left')
plt.tight_layout()
plt.savefig('farm_a_multi_sensor_overlay.png', dpi=150, bbox_inches='tight')
plt.show()
print('💾 Saved: farm_a_multi_sensor_overlay.png')

In [ ]:
# Plot 2: Z-score standardization — reveals subtle deviations invisible in raw data
fig, axes = plt.subplots(2, 1, figsize=(15, 9), sharex=True)
fig.suptitle('Farm A — Z-Score Standardization Reveals Subtle Pre-Fault Deviations\n'
             'All sensors on same scale — spikes show simultaneous anomalies',
             fontsize=12, fontweight='bold')

# Top: raw rolling means
for (bare, col), color in zip(SENSORS_A.items(), colors_a):
    if col not in df_a.columns:
        continue
    label   = get_label(feat_a, bare)[:28]
    rolling = df_a[col].rolling(window=DRIFT_WINDOW, min_periods=DRIFT_WINDOW//2).mean()
    axes[0].plot(df_a.index, rolling, linewidth=1.5, color=color, label=label)

axes[0].axvline(x=train_end_a, color='black', linestyle='--', linewidth=1.5)
axes[0].set_ylabel('Temperature (°C)')
axes[0].set_title('Raw 7-day rolling means', fontsize=9, fontweight='bold')
axes[0].legend(fontsize=7, loc='upper left', ncol=2)

# Bottom: z-scores
for (bare, col), color in zip(SENSORS_A.items(), colors_a):
    if col not in df_a.columns:
        continue
    label  = get_label(feat_a, bare)[:28]
    zscore = rolling_zscore(df_a[col], DRIFT_WINDOW)
    axes[1].plot(df_a.index, zscore, linewidth=1.0, color=color, alpha=0.8, label=label)

axes[1].axhline(y=3,  color='red', linestyle='--', linewidth=1.2, label='+3σ threshold')
axes[1].axhline(y=-3, color='red', linestyle='--', linewidth=1.2)
axes[1].axhline(y=0,  color='gray', linestyle=':', linewidth=0.8)
axes[1].axvline(x=train_end_a, color='black', linestyle='--', linewidth=1.5)
axes[1].set_ylabel('Z-score (standard deviations)')
axes[1].set_xlabel('Time step (10-min intervals)')
axes[1].set_title('Z-score standardized — subtle deviations become visible', fontsize=9, fontweight='bold')
axes[1].legend(fontsize=7, loc='upper left', ncol=2)
axes[1].set_ylim(-8, 8)

plt.tight_layout()
plt.savefig('farm_a_zscore_multisensor.png', dpi=150, bbox_inches='tight')
plt.show()
print('💾 Saved: farm_a_zscore_multisensor.png')

In [ ]:
# Plot 3: Anomaly heatmap — shows WHEN each sensor deviates
# Resample to daily resolution for readability
THRESHOLD = 2.5
heatmap_data_a = {}

for bare, col in SENSORS_A.items():
    if col not in df_a.columns:
        continue
    label  = get_label(feat_a, bare)[:32]
    zscore = rolling_zscore(df_a[col], DRIFT_WINDOW)
    # Flag timesteps exceeding threshold
    flagged = (zscore.abs() > THRESHOLD).astype(int)
    # Group into blocks of 144 (1 day) and take mean
    daily   = flagged.groupby(flagged.index // 144).mean()
    heatmap_data_a[label] = daily

heatmap_df_a = pd.DataFrame(heatmap_data_a).T

fig, ax = plt.subplots(figsize=(16, 4))
sns.heatmap(heatmap_df_a, cmap='Reds', ax=ax, cbar_kws={'label': 'Fraction of timesteps flagged'},
            linewidths=0, vmin=0, vmax=0.5)

# Mark fault window
fault_day = train_end_a // 144
ax.axvline(x=fault_day, color='black', linewidth=2, label='Fault window start')
ax.set_title('Farm A — Anomaly Heatmap: Which Sensors Flag Anomalies Each Day?\n'
             f'(Z-score > {THRESHOLD}σ — darker = more timesteps flagged that day)',
             fontweight='bold')
ax.set_xlabel('Day number')
ax.set_ylabel('')
plt.tight_layout()
plt.savefig('farm_a_anomaly_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('💾 Saved: farm_a_anomaly_heatmap.png')

---
## Section 3 — Farm B: Multi-Sensor Deviation Analysis
Transformer cell overheating — transformer sensors + generator bearings.

In [ ]:
farm_b  = farms['B']
feat_b  = farm_b['feat_desc']
ei_b    = farm_b['event_info']

transformer_b_id = ei_b[
    ei_b['event_description'].str.contains('transformer', case=False, na=False)
]['event_id'].iloc[0]

df_b      = load_event(farm_b, transformer_b_id)
train_end_b = df_b[df_b['train_test'] == 'train'].index.max()

SENSORS_B = {
    'sensor_47': 'sensor_47_avg',  # Transformer cell temp — primary
    'sensor_40': 'sensor_40_avg',  # Transformator core
    'sensor_41': 'sensor_41_avg',  # Transformer L1 mid-voltage
    'sensor_43': 'sensor_43_avg',  # Transformer L2 mid-voltage
    'sensor_45': 'sensor_45_avg',  # Transformer L3 mid-voltage
}

print('Farm B sensors for 2b analysis:')
for bare, col in SENSORS_B.items():
    print(f'  {col}: {get_label(feat_b, bare)}')

In [ ]:
# Rolling mean overlay Farm B
colors_b = ['tomato', 'darkorange', 'purple', 'steelblue', 'seagreen']

fig, ax = plt.subplots(figsize=(15, 6))
for (bare, col), color in zip(SENSORS_B.items(), colors_b):
    if col not in df_b.columns:
        continue
    label   = get_label(feat_b, bare)
    rolling = df_b[col].rolling(window=DRIFT_WINDOW, min_periods=DRIFT_WINDOW//2).mean()
    ax.plot(df_b.index, rolling, linewidth=1.8, color=color,
            label=f'{label[:35]}')

ax.axvline(x=train_end_b, color='black', linestyle='--',
           linewidth=1.5, label='Fault window start')
ax.set_title('Farm B — Transformer Cell Overheating: 7-Day Rolling Mean\nMultiple Transformer Sensors Overlaid',
             fontweight='bold')
ax.set_xlabel('Time step (10-min intervals)')
ax.set_ylabel('Temperature (°C)')
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig('farm_b_multi_sensor_overlay.png', dpi=150, bbox_inches='tight')
plt.show()
print('💾 Saved: farm_b_multi_sensor_overlay.png')

In [ ]:
# Z-score Farm B
fig, axes = plt.subplots(2, 1, figsize=(15, 9), sharex=True)
fig.suptitle('Farm B — Z-Score Reveals Subtle Transformer Temperature Deviations',
             fontsize=12, fontweight='bold')

for (bare, col), color in zip(SENSORS_B.items(), colors_b):
    if col not in df_b.columns:
        continue
    label   = get_label(feat_b, bare)[:30]
    rolling = df_b[col].rolling(window=DRIFT_WINDOW, min_periods=DRIFT_WINDOW//2).mean()
    zscore  = rolling_zscore(df_b[col], DRIFT_WINDOW)

    axes[0].plot(df_b.index, rolling, linewidth=1.5, color=color, label=label)
    axes[1].plot(df_b.index, zscore,  linewidth=1.0, color=color, alpha=0.8, label=label)

for ax in axes:
    ax.axvline(x=train_end_b, color='black', linestyle='--', linewidth=1.5)

axes[1].axhline(y=3,  color='red', linestyle='--', linewidth=1.2, label='+3σ')
axes[1].axhline(y=-3, color='red', linestyle='--', linewidth=1.2)
axes[1].axhline(y=0,  color='gray', linestyle=':', linewidth=0.8)
axes[1].set_ylim(-8, 8)

axes[0].set_ylabel('Temperature (°C)')
axes[0].set_title('Raw 7-day rolling means', fontsize=9, fontweight='bold')
axes[0].legend(fontsize=7, ncol=2)
axes[1].set_ylabel('Z-score')
axes[1].set_xlabel('Time step (10-min intervals)')
axes[1].set_title('Z-score standardized', fontsize=9, fontweight='bold')
axes[1].legend(fontsize=7, ncol=2)

plt.tight_layout()
plt.savefig('farm_b_zscore_multisensor.png', dpi=150, bbox_inches='tight')
plt.show()
print('💾 Saved: farm_b_zscore_multisensor.png')

In [ ]:
# Anomaly heatmap Farm B
heatmap_data_b = {}
for bare, col in SENSORS_B.items():
    if col not in df_b.columns:
        continue
    label   = get_label(feat_b, bare)[:32]
    zscore  = rolling_zscore(df_b[col], DRIFT_WINDOW)
    flagged = (zscore.abs() > THRESHOLD).astype(int)
    daily   = flagged.groupby(flagged.index // 144).mean()
    heatmap_data_b[label] = daily

heatmap_df_b = pd.DataFrame(heatmap_data_b).T

fig, ax = plt.subplots(figsize=(16, 4))
sns.heatmap(heatmap_df_b, cmap='Reds', ax=ax,
            cbar_kws={'label': 'Fraction of timesteps flagged'},
            linewidths=0, vmin=0, vmax=0.5)
fault_day_b = train_end_b // 144
ax.axvline(x=fault_day_b, color='black', linewidth=2)
ax.set_title('Farm B — Anomaly Heatmap: Transformer Sensors Day-by-Day\n'
             f'(Z-score > {THRESHOLD}σ — darker = more flagged timesteps)',
             fontweight='bold')
ax.set_xlabel('Day number')
plt.tight_layout()
plt.savefig('farm_b_anomaly_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('💾 Saved: farm_b_anomaly_heatmap.png')

---
## Section 4 — Farm C: Multi-Sensor Deviation Analysis

In [ ]:
farm_c  = farms['C']
feat_c  = farm_c['feat_desc']
ei_c    = farm_c['event_info']

anomaly_c = ei_c[ei_c['event_label'] == 'anomaly'].reset_index(drop=True)
df_c      = load_event(farm_c, anomaly_c['event_id'].iloc[0])
train_end_c = df_c[df_c['train_test'] == 'train'].index.max()

# Find top 5 available temperature sensors
temp_c = feat_c[
    feat_c['description'].str.contains('temp|temperature|heat', case=False, na=False)
]
available_c = [
    s for s in temp_c['sensor_name'].tolist()
    if f'{s}_avg' in df_c.columns
][:5]

SENSORS_C = {s: f'{s}_avg' for s in available_c}

print('Farm C sensors for 2b analysis:')
for bare, col in SENSORS_C.items():
    print(f'  {col}: {get_label(feat_c, bare)}')

In [ ]:
# Rolling mean overlay Farm C
colors_c = ['tomato', 'darkorange', 'purple', 'steelblue', 'seagreen']

fig, ax = plt.subplots(figsize=(15, 6))
for (bare, col), color in zip(SENSORS_C.items(), colors_c):
    label   = get_label(feat_c, bare)
    rolling = df_c[col].rolling(window=DRIFT_WINDOW, min_periods=DRIFT_WINDOW//2).mean()
    ax.plot(df_c.index, rolling, linewidth=1.8, color=color, label=f'{label[:35]}')

ax.axvline(x=train_end_c, color='black', linestyle='--',
           linewidth=1.5, label='Fault window start')
ax.set_title('Farm C — Fault Event: 7-Day Rolling Mean\nMultiple Thermal Sensors Overlaid',
             fontweight='bold')
ax.set_xlabel('Time step (10-min intervals)')
ax.set_ylabel('Temperature (°C)')
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig('farm_c_multi_sensor_overlay.png', dpi=150, bbox_inches='tight')
plt.show()
print('💾 Saved: farm_c_multi_sensor_overlay.png')

In [ ]:
# Z-score Farm C
fig, axes = plt.subplots(2, 1, figsize=(15, 9), sharex=True)
fig.suptitle('Farm C — Z-Score Reveals Subtle Multi-Sensor Deviations Before Fault',
             fontsize=12, fontweight='bold')

for (bare, col), color in zip(SENSORS_C.items(), colors_c):
    label   = get_label(feat_c, bare)[:30]
    rolling = df_c[col].rolling(window=DRIFT_WINDOW, min_periods=DRIFT_WINDOW//2).mean()
    zscore  = rolling_zscore(df_c[col], DRIFT_WINDOW)
    axes[0].plot(df_c.index, rolling, linewidth=1.5, color=color, label=label)
    axes[1].plot(df_c.index, zscore,  linewidth=1.0, color=color, alpha=0.8, label=label)

for ax in axes:
    ax.axvline(x=train_end_c, color='black', linestyle='--', linewidth=1.5)

axes[1].axhline(y=3,  color='red', linestyle='--', linewidth=1.2, label='+3σ')
axes[1].axhline(y=-3, color='red', linestyle='--', linewidth=1.2)
axes[1].axhline(y=0,  color='gray', linestyle=':', linewidth=0.8)
axes[1].set_ylim(-8, 8)
axes[0].set_ylabel('Temperature (°C)')
axes[0].set_title('Raw 7-day rolling means', fontsize=9, fontweight='bold')
axes[0].legend(fontsize=7, ncol=2)
axes[1].set_ylabel('Z-score')
axes[1].set_xlabel('Time step (10-min intervals)')
axes[1].set_title('Z-score standardized', fontsize=9, fontweight='bold')
axes[1].legend(fontsize=7, ncol=2)

plt.tight_layout()
plt.savefig('farm_c_zscore_multisensor.png', dpi=150, bbox_inches='tight')
plt.show()
print('💾 Saved: farm_c_zscore_multisensor.png')

In [ ]:
# Anomaly heatmap Farm C
heatmap_data_c = {}
for bare, col in SENSORS_C.items():
    label   = get_label(feat_c, bare)[:32]
    zscore  = rolling_zscore(df_c[col], DRIFT_WINDOW)
    flagged = (zscore.abs() > THRESHOLD).astype(int)
    daily   = flagged.groupby(flagged.index // 144).mean()
    heatmap_data_c[label] = daily

heatmap_df_c = pd.DataFrame(heatmap_data_c).T

fig, ax = plt.subplots(figsize=(16, 4))
sns.heatmap(heatmap_df_c, cmap='Reds', ax=ax,
            cbar_kws={'label': 'Fraction of timesteps flagged'},
            linewidths=0, vmin=0, vmax=0.5)
fault_day_c = train_end_c // 144
ax.axvline(x=fault_day_c, color='black', linewidth=2)
ax.set_title('Farm C — Anomaly Heatmap: Thermal Sensors Day-by-Day\n'
             f'(Z-score > {THRESHOLD}σ)', fontweight='bold')
ax.set_xlabel('Day number')
plt.tight_layout()
plt.savefig('farm_c_anomaly_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('💾 Saved: farm_c_anomaly_heatmap.png')

---
## Section 5 — Summary

In [ ]:
print('=' * 65)
print('QUESTION 2b — KEY FINDINGS')
print('How do visualizations highlight subtle multi-sensor deviations?')
print('=' * 65)
print()
print('Three visualization techniques applied across all farms:')
print()
print('1. ROLLING MEAN OVERLAY')
print('   Smooths 10-min noise into weekly trends.')
print('   Shows which sensors drift together before faults.')
print()
print('2. Z-SCORE STANDARDIZATION')
print('   Puts all sensors on the same scale.')
print('   Reveals subtle deviations invisible in raw °C values.')
print('   Sensors deviating together = stronger fault signal.')
print()
print('3. ANOMALY HEATMAP')
print('   Shows WHEN each sensor flags anomalies day by day.')
print('   Dark clusters near fault window = pre-fault warning pattern.')
print('   Multiple sensors darkening together = high confidence alert.')
print()
print('Key finding across all three farms:')
print('  Subtle deviations that are invisible in raw data become')
print('  clearly visible once z-score standardized and overlaid.')
print('  The heatmap is the most operator-friendly format.')
print('=' * 65)